In [ ]:
import pandas as pd

df = pd.read_csv("../dataset/restaurant_orders.csv")
print("Before cleanup:", df.shape)

df_clean = df[~df["visit_frequency"].isin(["string", "Weekly"])]
df_clean = df_clean[df_clean["age_group"] != "string"]
df_clean = df_clean[df_clean["gender"] != "string"]

print("After cleanup:", df_clean.shape)

df_clean.to_csv("../dataset/restaurant_orders.csv", index=False)
print("Saved cleaned dataset")

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer
import joblib
import os

df = pd.read_csv("../dataset/restaurant_orders.csv")
print(df.shape)
df.head()

(8000, 18)


,customer_id,age_group,gender,visit_frequency,preferred_cuisine,favorite_food_category,previous_orders,frequently_ordered_item,ordered_item,average_bill_amount,time_of_visit,day_of_week,season,veg_nonveg_pref,spice_preference,customer_rating,order_timestamp,ordered_again
0,2032,18-25,Other,Frequent,Italian,Main Course,15,Margherita Pizza,Margherita Pizza,634.34,Lunch,Thursday,Monsoon,Veg,Medium,3.5,2025-08-07 14:10:19,1
1,1126,60+,Female,Frequent,Continental,Bread,32,Club Sandwich,Club Sandwich,424.60,Dinner,Wednesday,Winter,Non-Veg,Mild,3.4,2025-08-07 14:39:26,1
2,1992,26-35,Male,Regular,Italian,Starter,16,Alfredo Pasta,Alfredo Pasta,571.12,Breakfast,Wednesday,Winter,Veg,Medium,4.7,2025-08-07 15:43:22,1
3,2111,26-35,Female,Regular,North Indian,Main Course,8,Paneer Butter Masala,Paneer Butter Masala,462.69,Dinner,Sunday,Winter,Veg,Medium,3.9,2025-08-07 17:38:34,1
4,2290,46-60,Male,Regular,South Indian,Main Course,11,Idli Sambar,Idli Sambar,547.02,Lunch,Sunday,Summer,Non-Veg,Medium,4.0,2025-08-07 18:20:02,1


In [2]:
# Build a per-customer profile: their preferences + dish interaction history
customer_profile = df.groupby("customer_id").agg({
    "preferred_cuisine": lambda x: x.mode()[0],
    "favorite_food_category": lambda x: x.mode()[0],
    "veg_nonveg_pref": lambda x: x.mode()[0],
    "spice_preference": lambda x: x.mode()[0],
    "visit_frequency": lambda x: x.mode()[0],
    "customer_rating": "mean",
    "average_bill_amount": "mean"
}).reset_index()

print(customer_profile.shape)
customer_profile.head()

(1441, 8)


,customer_id,preferred_cuisine,favorite_food_category,veg_nonveg_pref,spice_preference,visit_frequency,customer_rating,average_bill_amount
0,1000,Mughlai,Bread,Veg,Mild,Regular,3.471429,574.344286
1,1001,Mughlai,Main Course,Veg,Medium,First-time,4.333333,495.266667
2,1002,South Indian,Bread,Veg,Mild,Regular,3.366667,667.166667
3,1003,Continental,Main Course,Non-Veg,Spicy,First-time,4.900000,652.570000
4,1004,South Indian,Beverage,Veg,Mild,Regular,3.300000,454.350000


In [3]:
# One-hot encode categorical features so we can compute similarity between customers
profile_encoded = pd.get_dummies(
    customer_profile[["preferred_cuisine", "favorite_food_category",
                       "veg_nonveg_pref", "spice_preference", "visit_frequency"]]
)
profile_encoded["customer_rating"] = customer_profile["customer_rating"]
profile_encoded["average_bill_amount"] = customer_profile["average_bill_amount"]

# Compute similarity between every pair of customers (collaborative filtering base)
customer_similarity = cosine_similarity(profile_encoded)
customer_sim_df = pd.DataFrame(customer_similarity,
                                index=customer_profile["customer_id"],
                                columns=customer_profile["customer_id"])
print(customer_sim_df.shape)

(1441, 1441)


In [7]:
def recommend_dishes(customer_id, top_n=5):
    if customer_id not in customer_profile["customer_id"].values:
        return {"error": "Customer not found"}

    cust_row = customer_profile[customer_profile["customer_id"] == customer_id].iloc[0]
    cust_orders = df[df["customer_id"] == customer_id]

    # --- Content-based score: score all dishes ever ordered in the dataset ---
    all_dishes = df["ordered_item"].unique()
    dish_scores = {}

    for dish in all_dishes:
        dish_rows = df[df["ordered_item"] == dish]
        content_score = 0

        # cuisine match (infer dish's cuisine from majority context)
        if (dish_rows["preferred_cuisine"] == cust_row["preferred_cuisine"]).mean() > 0.3:
            content_score += 0.35
        if (dish_rows["favorite_food_category"] == cust_row["favorite_food_category"]).any():
            content_score += 0.2
        if (dish_rows["veg_nonveg_pref"] == cust_row["veg_nonveg_pref"]).mean() > 0.5:
            content_score += 0.15
        if (dish_rows["spice_preference"] == cust_row["spice_preference"]).mean() > 0.3:
            content_score += 0.1

        # --- Collaborative score: weighted by similarity to other customers who ordered this dish ---
        collab_score = 0
        orderers = dish_rows["customer_id"].unique()
        sims = [customer_sim_df.loc[customer_id, o] for o in orderers if o != customer_id and o in customer_sim_df.columns]
        if sims:
            collab_score = np.mean(sims) * 0.3

        # --- Rating boost ---
        avg_rating = dish_rows["customer_rating"].mean()
        rating_score = (avg_rating / 5) * 0.2

        # --- Recency: already ordered this exact dish? small boost if they liked it ---
        already_ordered = dish in cust_orders["ordered_item"].values
        recency_score = 0.15 if already_ordered and cust_orders["ordered_again"].mean() > 0.5 else 0

        total = content_score + collab_score + rating_score + recency_score
        dish_scores[dish] = {
            "score": total,
            "content_score": content_score,
            "collab_score": collab_score,
            "rating_score": rating_score,
            "recency_score": recency_score,
            "avg_rating": round(avg_rating, 2)
        }

    ranked = sorted(dish_scores.items(), key=lambda x: x[1]["score"], reverse=True)[:top_n]

    results = []
    max_score = max(d[1]["score"] for d in ranked) if ranked else 1

    for dish, s in ranked:
        confidence = round(float(min(s["score"] / max_score * 100, 99.9)), 1)

        # pick the strongest signal as the reason (not just the first match)
        signal_strengths = {
            "Frequently ordered by you before": s["recency_score"],
            "Highly rated by customers with similar preferences": s["rating_score"],
            "Customers with similar taste preferred this": s["collab_score"],
            "Matches your preferred cuisine": s["content_score"],
        }
        best_reason = max(signal_strengths, key=signal_strengths.get)
        if signal_strengths[best_reason] == 0:
            best_reason = "Complements your recent orders"

        results.append({
            "food": dish,
            "confidence": confidence,
            "reason": best_reason
        })

    return {"customer_id": int(customer_id), "recommendations": results}

In [10]:
def recommend_dishes(customer_id, top_n=5):
    if customer_id not in customer_profile["customer_id"].values:
        return {"error": "Customer not found"}

    cust_row = customer_profile[customer_profile["customer_id"] == customer_id].iloc[0]
    cust_orders = df[df["customer_id"] == customer_id]

    # --- Content-based score: score all dishes ever ordered in the dataset ---
    all_dishes = df["ordered_item"].unique()
    dish_scores = {}

    for dish in all_dishes:
        dish_rows = df[df["ordered_item"] == dish]
        content_score = 0

        # cuisine match (infer dish's cuisine from majority context)
        if (dish_rows["preferred_cuisine"] == cust_row["preferred_cuisine"]).mean() > 0.3:
            content_score += 0.35
        if (dish_rows["favorite_food_category"] == cust_row["favorite_food_category"]).any():
            content_score += 0.2
        if (dish_rows["veg_nonveg_pref"] == cust_row["veg_nonveg_pref"]).mean() > 0.5:
            content_score += 0.15
        if (dish_rows["spice_preference"] == cust_row["spice_preference"]).mean() > 0.3:
            content_score += 0.1

        # --- Collaborative score: weighted by similarity to other customers who ordered this dish ---
        collab_score = 0
        orderers = dish_rows["customer_id"].unique()
        sims = [customer_sim_df.loc[customer_id, o] for o in orderers if o != customer_id and o in customer_sim_df.columns]
        if sims:
            collab_score = np.mean(sims) * 0.3

        # --- Rating boost ---
        avg_rating = dish_rows["customer_rating"].mean()
        rating_score = (avg_rating / 5) * 0.2

        # --- Recency: already ordered this exact dish? small boost if they liked it ---
        already_ordered = dish in cust_orders["ordered_item"].values
        recency_score = 0.15 if already_ordered and cust_orders["ordered_again"].mean() > 0.5 else 0

        total = content_score + collab_score + rating_score + recency_score
        dish_scores[dish] = {
            "score": total,
            "content_score": content_score,
            "collab_score": collab_score,
            "rating_score": rating_score,
            "recency_score": recency_score,
            "avg_rating": round(avg_rating, 2)
        }

    ranked = sorted(dish_scores.items(), key=lambda x: x[1]["score"], reverse=True)[:top_n]

    results = []
    max_score = max(d[1]["score"] for d in ranked) if ranked else 1

    for dish, s in ranked:
        confidence = round(float(min(s["score"] / max_score * 100, 99.9)), 1)

        # normalize each signal to 0-1 by its max possible value, so no signal always wins by default
        signal_strengths = {
            "Frequently ordered by you before": s["recency_score"] / 0.15,
            "Highly rated by customers with similar preferences": s["rating_score"] / 0.2,
            "Customers with similar taste preferred this": s["collab_score"] / 0.3,
            "Matches your preferred cuisine": s["content_score"] / 0.8,
        }
        best_reason = max(signal_strengths, key=signal_strengths.get)
        if signal_strengths[best_reason] == 0:
            best_reason = "Complements your recent orders"

        results.append({
            "food": dish,
            "confidence": confidence,
            "reason": best_reason
        })

    return {"customer_id": int(customer_id), "recommendations": results}

In [11]:
sample_id = customer_profile["customer_id"].iloc[0]
recommend_dishes(sample_id)

{'customer_id': 1000,
 'recommendations': [{'food': 'Kebabs',
   'confidence': 99.9,
   'reason': 'Frequently ordered by you before'},
  {'food': 'Shahi Paneer',
   'confidence': 82.1,
   'reason': 'Frequently ordered by you before'},
  {'food': 'Naan',
   'confidence': 81.9,
   'reason': 'Frequently ordered by you before'},
  {'food': 'Mutton Rogan Josh',
   'confidence': 78.5,
   'reason': 'Customers with similar taste preferred this'},
  {'food': 'Chicken Biryani',
   'confidence': 71.3,
   'reason': 'Customers with similar taste preferred this'}]}

In [12]:
import joblib

joblib.dump({
    "customer_profile": customer_profile,
    "customer_sim_df": customer_sim_df,
    "df": df
}, "../trained_models/recommendation_model.pkl")

print("Model saved successfully")

Model saved successfully


In [14]:
def evaluate_model(k=5, sample_size=200):
    """
    Precision@K, Recall@K, F1@K evaluated over a sample of customers.
    Ground truth = dishes the customer rated 4+ or marked 'ordered_again'.
    """
    eval_customers = customer_profile["customer_id"].sample(
        min(sample_size, len(customer_profile)), random_state=42
    )

    precisions, recalls = [], []

    for cust_id in eval_customers:
        # Ground truth: dishes this customer actually liked
        cust_data = df[df["customer_id"] == cust_id]
        liked_dishes = set(
            cust_data[(cust_data["customer_rating"] >= 4) | (cust_data["ordered_again"] == 1)]["ordered_item"]
        )

        if not liked_dishes:
            continue

        # Model's recommendations
        rec_result = recommend_dishes(cust_id, top_n=k)
        recommended_dishes = set(r["food"] for r in rec_result.get("recommendations", []))

        hits = recommended_dishes & liked_dishes

        precision = len(hits) / k
        recall = len(hits) / len(liked_dishes)

        precisions.append(precision)
        recalls.append(recall)

    avg_precision = np.mean(precisions)
    avg_recall = np.mean(recalls)
    f1 = (2 * avg_precision * avg_recall / (avg_precision + avg_recall)
          if (avg_precision + avg_recall) > 0 else 0)

    return {
        "Precision@K": round(float(avg_precision), 4),
        "Recall@K": round(float(avg_recall), 4),
        "F1 Score": round(float(f1), 4),
        "Evaluated on": len(precisions)
    }

results = evaluate_model(k=5, sample_size=200)
print(results)

{'Precision@K': 0.4884, 'Recall@K': 0.8382, 'F1 Score': 0.6172, 'Evaluated on': 190}
